In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# RBFE single pair (BRD-2 vs BRD-3)

End-to-end workflow for one relative binding free energy pair:

1. Load the BRD protein and two ligands, register them on the data platform
2. Run system prep in RBFE mode
3. Inspect the prepared system
4. Run RBFE FEP on the prepared system

## Setup

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Ligand,
    Protein,
    RBFE,
    RBFEParams,
    SystemPrep,
    PreparedSystem,
)
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

## 1. Load structures and register on the data platform

We use the BRD4 example protein and two congeneric ligands from the bundled
dataset. `sync()` uploads files (when needed) and registers records on the
data platform.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync()
protein.id

In [ ]:
ligand1 = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand1.sync()

ligand2 = Ligand.from_sdf(BRD_DATA_DIR / "brd-3.sdf")
ligand2.sync()

ligand1, ligand2

## 2. System prep (RBFE mode)

Pass both ligands to `SystemPrep` to prepare binding and solvation legs for
the pair. This runs synchronously via `deeporigin.system-prep`.

In [ ]:
systems = PreparedSystem.from_result(ligand1_id=ligand1.id, ligand2_id=ligand2.id, protein_id=protein.id)
if len(systems) == 0:
    sysprep = SystemPrep(
    protein=protein,
    ligand1=ligand1,
    ligand2=ligand2,
    )
    system = sysprep.run()
else:
    system = systems[0]
system

## 3. Show the prepared system

Visualize the solvated system PDB returned by system prep.

In [ ]:
system.show()

## 4. Run RBFE on the prepared system

Submit FEP only (`mode="rbfe"`) using the prepared binding/solvation XML paths.
We quote first, then confirm to start the job.

`test_run=1` shortens the simulation for exploration; use `test_run=0` for
production-quality results.

In [ ]:
rbfe = RBFE(
    prepared_systems=[system],
    params=RBFEParams(test_run=1),
)
rbfe

In [ ]:
rbfe.start(quote=True)
rbfe.estimate

In [ ]:
rbfe.confirm()

In [ ]:
task = await rbfe.watch()

## Results

In [ ]:
rbfe.get_results()

In [ ]:
rbfe = RBFE.from_id("a5484958-059f-4b1b-ba2c-664adf23e8e8")

In [ ]:
rbfe.get_results()